# CNN 混凝土裂缝识别训练与应用

本 Notebook 保留当前翻转增强实验：原始 ConcreteCrackCNN、加权交叉熵、学习率 0.0001、最多 30 轮、验证 F1 连续 5 轮未改善时早停。标签固定为无裂缝 uncracked=0、有裂缝 cracked=1。

**运行路线**

- 训练新模型：依次运行第 1–16 节，再运行第 17–19 节查看最佳模型表现。第 14 节会重新训练，并覆盖当前输出目录中的同名模型文件；新实验请先改第 2 节的目录。
- 只预测自己的图片：运行第 1、2、3、4、9、17、21 节即可，无需训练或加载整个数据集。第 17 节读取已有的 model_best.pt。
- 最终测试：方案确定后，在第 2 节启用 RUN_TEST_EVALUATION，再运行第 20 节。
- 小样本排错：第 22 节为可选诊断，默认跳过；不代表实际泛化性能。

整理前的全部代码、历史输出和执行序号保存在同目录 _backups 文件夹。当前显示输出已清空，避免把旧实验结果误当成新顺序的执行结果。

## 1 导入依赖与定义类别名称

用途：导入后续代码需要的库，并固定类别顺序。

运行时机：每次重启内核后首先运行。输出：PyTorch 版本和 CUDA 是否可用。

In [ ]:
# 1 导入依赖与定义类别名称
# 用途和运行条件见上方说明单元。

from collections import Counter
from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

CLASS_NAMES = ('uncracked', 'cracked')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2 设置路径、训练参数与可选检查

用途：集中设置数据位置、实验输出目录和训练参数。PROJECT_ROOT 是内核当前工作目录，不保证始终等于 Notebook 所在目录；路径不对时可改成绝对路径。

当前使用完整分组数据；QUICK_RUN 和 MAX_TRAIN_SAMPLES/MAX_TEST_SAMPLES 是旧版本保留变量，不再自动启用小样本模式。EPOCHS 是轮数上限，PATIENCE 是早停等待轮数。

输出：路径、学习率和轮数上限。只预测时，OUTPUT_DIR 应指向已有模型的实验目录。

In [ ]:
# 2 设置路径、训练参数与可选检查
# 用途和运行条件见上方说明单元。

PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = (PROJECT_ROOT / 'SDNET2018' / 'SDNET2018').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'group_validation_flip'

QUICK_RUN = False
SEED = 42
LEARNING_RATE = 0.0001
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 5
RUN_TEST_EVALUATION = False
RUN_SMALL_SAMPLE_CHECK = False
NUM_WORKERS = 0
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None
MAX_TRAIN_BATCHES = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Data root:', DATA_ROOT)
print('Output directory:', OUTPUT_DIR)
print('Learning rate:', LEARNING_RATE)
print('Epochs:', EPOCHS)

## 3 固定随机种子并选择计算设备

用途：设置 Python、NumPy、PyTorch 的随机起点，自动选择 CUDA 或 CPU。固定种子有助于重复实验，但不能保证跨环境结果完全相同。

依赖：第 1、2 节。输出：设备和显卡名称。

In [ ]:
# 3 固定随机种子并选择计算设备
# 用途和运行条件见上方说明单元。

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 4 定义图像预处理与标签规则

用途：训练图片随机水平/垂直翻转；验证、测试和预测只使用固定预处理。所有图片转为 224×224 的 RGB Tensor 并标准化。

C 开头的目录映射为裂缝 1，U 开头映射为无裂缝 0。此处只定义处理规则和 Dataset 类，还没有加载图片列表。

In [ ]:
# 4 定义图像预处理与标签规则
# 用途和运行条件见上方说明单元。

fixed_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 验证、测试和单张预测统一使用固定预处理
transform = fixed_transform


def semantic_label(folder_name):
    prefix = folder_name.strip().upper()[:1]
    if prefix == 'C':
        return 1  # cracked
    if prefix == 'U':
        return 0  # uncracked
    raise ValueError(f'不支持的类别目录：{folder_name}')


class CrackImageFolder(datasets.ImageFolder):
    def __init__(self, root, transform=None):
        super().__init__(str(root), transform=transform)
        remapped = [
            (path, semantic_label(Path(path).parent.name))
            for path, _ in self.samples
        ]
        self.samples = remapped
        self.imgs = remapped
        self.targets = [target for _, target in remapped]
        self.classes = list(CLASS_NAMES)
        self.class_to_idx = {'uncracked': 0, 'cracked': 1}

## 5 加载数据列表并检查类别数量

用途：分别创建训练增强版、训练固定版和测试版数据集，检查前两者图片顺序一致。

依赖：第 1–4 节及 DATA_ROOT 下的 train/test 目录。输出：两类图片数量；不会复制或移动原始图片。

In [ ]:
# 5 加载数据列表并检查类别数量
# 用途和运行条件见上方说明单元。

assert (DATA_ROOT / 'train').is_dir(), f'找不到训练集：{DATA_ROOT / "train"}'
assert (DATA_ROOT / 'test').is_dir(), f'找不到测试集：{DATA_ROOT / "test"}'

train_full = CrackImageFolder(DATA_ROOT / 'train', transform=fixed_transform)
train_aug_full = CrackImageFolder(DATA_ROOT / 'train', transform=train_transform)
test_full = CrackImageFolder(DATA_ROOT / 'test', transform=fixed_transform)

assert [path for path, _ in train_full.samples] == [
    path for path, _ in train_aug_full.samples
], '固定预处理和增强预处理的图片顺序不一致'


def summarize(dataset):
    counts = Counter(dataset.targets)
    return {CLASS_NAMES[index]: counts[index] for index in range(2)}

print('Train:', summarize(train_full), 'total =', len(train_full))
print('Train augmented:', summarize(train_aug_full), 'total =', len(train_aug_full))
print('Test :', summarize(test_full), 'total =', len(test_full))
print('Positive class for Recall/F1: cracked(1)')

## 6 查看图片文件名

用途：打印每类前 10 个文件名，帮助检查命名方式。

例如 7001-1.jpg 的分组前缀是 7001。仅检查文件名不能证明标签内容正确；需要时再查看图片。依赖：第 5 节。

In [ ]:
# 6 查看图片文件名
# 用途和运行条件见上方说明单元。

for label, name in enumerate(CLASS_NAMES):
    filenames = [
        Path(path).name
        for path, target in train_full.samples
        if target == label
    ]
    print(name, filenames[:10])

## 7 按文件名前缀划分训练集与验证集

用途：同一前缀的图片放在同一组，约 20% 的组用于验证。组数比例不等于精确图片比例。

依赖：第 5 节的数据列表。输出：train_indices、val_indices 和类别统计；检查索引不重叠且两边都有两个类别。固定 SEED 和文件列表后，后续实验复用相同划分。

In [ ]:
# 7 按文件名前缀划分训练集与验证集
# 用途和运行条件见上方说明单元。

# 1. 按文件名前缀分组
groups = {}

for index, (path, label) in enumerate(train_full.samples):
    group_id = Path(path).stem.split("-")[0]
    groups.setdefault(group_id, []).append(index)

# 2. 打乱组的顺序，抽取约 20% 的组作为验证集
group_ids = sorted(groups)
random.Random(SEED).shuffle(group_ids)

assert len(group_ids) >= 2, "分组数量不足，无法划分"

val_group_count = min(
    len(group_ids) - 1,
    max(1, round(len(group_ids) * 0.2))
)

val_groups = set(group_ids[:val_group_count])

# 3. 分别记录训练图片和验证图片的编号
train_indices = []
val_indices = []

for group_id, indices in groups.items():
    if group_id in val_groups:
        val_indices.extend(indices)
    else:
        train_indices.extend(indices)

# 4. 检查各部分的类别数量
def count_labels(indices):
    counts = Counter(train_full.targets[i] for i in indices)
    return {CLASS_NAMES[label]: counts[label] for label in range(2)}

print("总分组数：", len(groups))
print("验证集分组数：", len(val_groups))
print("训练集图片数：", len(train_indices), count_labels(train_indices))
print("验证集图片数：", len(val_indices), count_labels(val_indices))

assert set(train_indices).isdisjoint(val_indices)
assert len(train_indices) + len(val_indices) == len(train_full)

for indices in (train_indices, val_indices):
    assert {train_full.targets[i] for i in indices} == {0, 1}, \
        "某个集合缺少一个类别，需要调整分组划分"

## 8 创建训练、验证、测试与诊断加载器

用途：训练加载器打乱图片并使用翻转增强；验证和测试加载器不打乱、使用固定预处理。

另建 train_eval_loader，专门在不随机翻转的条件下诊断训练集，便于和验证集比较。创建测试加载器不会执行测试。

In [ ]:
# 8 创建训练、验证、测试与诊断加载器
# 用途和运行条件见上方说明单元。

# 训练使用翻转增强数据集，验证和测试使用固定预处理数据集
train_data = Subset(train_aug_full, train_indices)
val_data = Subset(train_full, val_indices)
test_data = test_full

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("参与训练：", len(train_loader.dataset))
print("用于验证：", len(val_loader.dataset))
print("用于测试：", len(test_loader.dataset))
print("训练预处理：水平翻转 + 垂直翻转")
print("验证/测试预处理：固定 Resize + ToTensor + Normalize")

train_eval_loader = DataLoader(
    Subset(train_full, train_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

## 9 定义 CNN 网络结构

用途：定义两层卷积、池化和全连接层。输入必须为 3×224×224，输出两类 logits。

本节只定义类，不创建训练模型，也不更新参数。只加载已有模型进行预测时也要先运行本节。

In [ ]:
# 9 定义 CNN 网络结构
# 用途和运行条件见上方说明单元。

class ConcreteCrackCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.relu3(self.fc1(x))
        return self.fc2(x)

## 10 定义单轮训练与验证函数

用途：train_one_epoch 执行反向传播和参数更新；evaluate 只检查结果，不更新参数。

两者均按类别权重总和汇总损失。验证返回 Precision、Recall、Specificity、F1、BA 和混淆矩阵计数。运行本节仅定义函数，不会开始训练。

In [ ]:
# 10 定义单轮训练与验证函数
# 用途和运行条件见上方说明单元。

def train_one_epoch(model, loader, criterion, optimizer, device, max_batches=None):
    model.train()
    weighted_loss_sum = 0.0
    weight_sum = 0.0
    for batch_index, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        batch_weights = (
            criterion.weight[labels]
            if criterion.weight is not None
            else torch.ones_like(labels, dtype=torch.float32)
        )
        weighted_loss_sum += loss.item() * batch_weights.sum().item()
        weight_sum += batch_weights.sum().item()
    return weighted_loss_sum / weight_sum


@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    weighted_loss_sum = 0.0
    weight_sum = 0.0
    targets, predictions = [], []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        batch_weights = (
            criterion.weight[labels]
            if criterion.weight is not None
            else torch.ones_like(labels, dtype=torch.float32)
        )
        weighted_loss_sum += loss.item() * batch_weights.sum().item()
        weight_sum += batch_weights.sum().item()
        predicted = logits.argmax(dim=1)
        targets.extend(labels.cpu().tolist())
        predictions.extend(predicted.cpu().tolist())

    true_positive = sum(t == 1 and p == 1 for t, p in zip(targets, predictions))
    false_positive = sum(t == 0 and p == 1 for t, p in zip(targets, predictions))
    false_negative = sum(t == 1 and p == 0 for t, p in zip(targets, predictions))
    true_negative = sum(t == 0 and p == 0 for t, p in zip(targets, predictions))

    def safe_divide(numerator, denominator):
        return numerator / denominator if denominator else 0.0

    precision = safe_divide(true_positive, true_positive + false_positive)
    recall = safe_divide(true_positive, true_positive + false_negative)
    specificity = safe_divide(true_negative, true_negative + false_positive)
    f1 = safe_divide(2 * precision * recall, precision + recall)
    metrics = {
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'balanced_accuracy': (recall + specificity) / 2,
        'true_positive': true_positive,
        'false_positive': false_positive,
        'false_negative': false_negative,
        'true_negative': true_negative,
    }
    return weighted_loss_sum / weight_sum, metrics

## 11 定义预测收集与分类指标函数

用途：收集整批预测，并计算以裂缝为正类的 Accuracy、Precision、Recall、F1、Specificity、BA。

这些是通用函数，供最终测试使用；运行本节本身不会访问测试集。

In [ ]:
# 11 定义预测收集与分类指标函数
# 用途和运行条件见上方说明单元。

@torch.inference_mode()
def collect_predictions(model, loader, device):
    model.eval()
    targets, predictions = [], []
    for images, labels in loader:
        logits = model(images.to(device))
        predicted = logits.argmax(dim=1).cpu()
        targets.extend(labels.tolist())
        predictions.extend(predicted.tolist())
    return targets, predictions


def safe_div(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def calculate_metrics(targets, predictions):
    tp = sum(t == 1 and p == 1 for t, p in zip(targets, predictions))
    fp = sum(t == 0 and p == 1 for t, p in zip(targets, predictions))
    fn = sum(t == 1 and p == 0 for t, p in zip(targets, predictions))
    tn = sum(t == 0 and p == 0 for t, p in zip(targets, predictions))
    accuracy = safe_div(tp + tn, len(targets))
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    f1 = safe_div(2 * precision * recall, precision + recall)
    return {
        'accuracy': accuracy,
        'balanced_accuracy': (recall + specificity) / 2,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'true_positive': tp,
        'false_positive': fp,
        'false_negative': fn,
        'true_negative': tn,
        'total': len(targets),
    }

## 12 定义概率分布诊断函数

用途：统计预测裂缝概率的均值、标准差、分位数及两类真实标签下的平均概率，用于判断模型是否总预测同一类。

依赖：第 10 节。此处只定义函数；第 18 节才实际执行诊断。

In [ ]:
# 12 定义概率分布诊断函数
# 用途和运行条件见上方说明单元。

@torch.inference_mode()
def collect_probability_diagnostics(model, loader, device):
    model.eval()
    targets, predictions, cracked_probabilities = [], [], []
    for images, labels in loader:
        probabilities = torch.softmax(model(images.to(device)), dim=1)
        predicted = probabilities.argmax(dim=1).cpu()
        targets.extend(labels.tolist())
        predictions.extend(predicted.tolist())
        cracked_probabilities.extend(probabilities[:, 1].cpu().tolist())
    return targets, predictions, np.array(cracked_probabilities)


def summarize_current_split(model, name, loader):
    loss, split_metrics = evaluate(model, loader, criterion, device)
    targets, predictions, cracked_probabilities = collect_probability_diagnostics(
        model, loader, device
    )
    summary = {
        'loss': loss,
        'metrics': split_metrics,
        'cracked_probability_mean': float(cracked_probabilities.mean()),
        'cracked_probability_std': float(cracked_probabilities.std()),
        'cracked_probability_quantiles': np.quantile(
            cracked_probabilities, [0, 0.25, 0.5, 0.75, 1]
        ).tolist(),
        'actual_uncracked_probability_mean': float(
            cracked_probabilities[np.array(targets) == 0].mean()
        ),
        'actual_cracked_probability_mean': float(
            cracked_probabilities[np.array(targets) == 1].mean()
        ),
    }
    print(f'[{name}] loss:', f"{loss:.6f}")
    print(f'[{name}] metrics:', split_metrics)
    print(f'[{name}] probability mean/std:', summary['cracked_probability_mean'], summary['cracked_probability_std'])
    print(f'[{name}] probability quantiles:', summary['cracked_probability_quantiles'])
    print(f'[{name}] actual-class probability means:', {
        'uncracked': summary['actual_uncracked_probability_mean'],
        'cracked': summary['actual_cracked_probability_mean'],
    })
    return summary

## 13 初始化新模型、类别权重和优化器

用途：按相同种子新建模型，用实际训练子集计算类别权重，并创建 Adam 优化器。

依赖：分组名单、第 9 节模型类。再次运行本节会重置内存中的模型和优化器；只预测时跳过本节。输出：结构、参数量、类别数量和权重。

In [ ]:
# 13 初始化新模型、类别权重和优化器
# 用途和运行条件见上方说明单元。

seed_everything(SEED)
model = ConcreteCrackCNN().to(device)
# 类别权重只根据实际参与训练的图片计算
train_labels = [train_full.targets[i] for i in train_indices]

class_counts = torch.bincount(
    torch.tensor(train_labels),
    minlength=2
).float()
class_weights = class_counts.sum() / (2 * class_counts)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('Class counts:', class_counts.tolist())
print('Class weights:', class_weights.tolist())

## 14 开始训练、每轮验证并保存最佳模型

用途：最多训练 EPOCHS 轮；每轮记录训练/验证结果。验证 F1 严格超过历史最佳时保存 model_best.pt；每轮保存 model_last.pt。

连续 PATIENCE 轮未改善就停止。最佳模型依据验证 F1 选出，不代表已满足应用要求。依赖：第 1–13 节；运行本节会写入当前实验目录。

In [ ]:
# 14 开始训练、每轮验证并保存最佳模型
# 用途和运行条件见上方说明单元。

history = {
    'train_loss': [],
    'val_loss': [],
    'val_precision': [],
    'val_recall': [],
    'val_specificity': [],
    'val_f1': [],
    'val_balanced_accuracy': [],
}
best_val_f1 = -1.0
best_epoch = 0
patience = PATIENCE
rounds_without_improvement = 0
started = time.perf_counter()

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, device, MAX_TRAIN_BATCHES
    )
    val_loss, val_metrics = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])
    history['val_specificity'].append(val_metrics['specificity'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_balanced_accuracy'].append(val_metrics['balanced_accuracy'])

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'architecture': 'ConcreteCrackCNN',
        'class_names': list(CLASS_NAMES),
        'positive_class': 'cracked',
        'image_size': [224, 224],
        'normalization': {
            'mean': [0.485, 0.456, 0.406],
            'std': [0.229, 0.224, 0.225],
        },
        'loss': 'CrossEntropyLoss',
        'class_counts': class_counts.tolist(),
        'class_weights': class_weights.tolist(),
        'epoch': epoch + 1,
        'val_metrics': val_metrics,
    }
    torch.save(checkpoint, OUTPUT_DIR / 'model_last.pt')

    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        best_epoch = epoch + 1
        rounds_without_improvement = 0
        torch.save(checkpoint, OUTPUT_DIR / 'model_best.pt')
    else:
        rounds_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/{EPOCHS}, "
        f"Train Loss: {train_loss:.4f}, "
        f"Val Loss: {val_loss:.4f}, "
        f"Val Precision: {val_metrics['precision']:.4f}, "
        f"Val Recall: {val_metrics['recall']:.4f}, "
        f"Val F1: {val_metrics['f1']:.4f}, "
        f"Val BA: {val_metrics['balanced_accuracy']:.4f}"
    )

    if rounds_without_improvement >= patience:
        print(f'Early stopping after {patience} rounds without validation F1 improvement.')
        break

elapsed_seconds = time.perf_counter() - started
epochs_ran = len(history['train_loss'])
print(f'Training time: {elapsed_seconds:.2f} seconds')
print(f'Epochs ran: {epochs_ran}')
print(f'Best epoch by validation F1: {best_epoch}')
print(f'Best validation F1: {best_val_f1:.4f}')

## 15 保存配置、划分名单和训练历史

用途：保存 config.json、split.json、history.json 和 metrics.json。metrics.json 汇总的是最佳验证结果，不是测试成绩。

依赖：第 14 节训练完成。本节不重复保存模型参数，避免加载最佳模型后误把它覆盖为“最后模型”。split.json 同时保留分组名单和图片索引。

In [ ]:
# 15 保存配置、划分名单和训练历史
# 用途和运行条件见上方说明单元。

best_index = best_epoch - 1
best_validation_metrics = {
    'precision': history['val_precision'][best_index],
    'recall': history['val_recall'][best_index],
    'specificity': history['val_specificity'][best_index],
    'f1': history['val_f1'][best_index],
    'balanced_accuracy': history['val_balanced_accuracy'][best_index],
}

epochs_ran = len(history['train_loss'])
# 两份模型已由训练循环保存；此处仅保存实验记录。


config = {
    'seed': SEED,
    'experiment': OUTPUT_DIR.name,
    'model': 'ConcreteCrackCNN',
    'loss': 'weighted_cross_entropy',
    'augmentation': ['RandomHorizontalFlip', 'RandomVerticalFlip'],
    'quick_run': QUICK_RUN,
    'epoch_limit': EPOCHS,
    'epochs_ran': epochs_ran,
    'early_stopping_patience': patience,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'device': str(device),
    'class_weights': class_weights.tolist(),
}
(OUTPUT_DIR / 'config.json').write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8'
)

split = {
    'seed': SEED,
    'train_indices': train_indices,
    'validation_indices': val_indices,
    'validation_groups': sorted(val_groups),
    'training_groups': sorted(set(groups) - val_groups),
    'train_count': len(train_indices),
    'validation_count': len(val_indices),
    'test_count': len(test_data),
    'train_labels': count_labels(train_indices),
    'validation_labels': count_labels(val_indices),
    'test_labels': summarize(test_full),
}
(OUTPUT_DIR / 'split.json').write_text(
    json.dumps(split, ensure_ascii=False, indent=2), encoding='utf-8'
)

(OUTPUT_DIR / 'history.json').write_text(
    json.dumps({
        'history': history,
        'best_epoch': best_epoch,
        'best_val_f1': best_val_f1,
        'epochs_ran': epochs_ran,
        'elapsed_seconds': elapsed_seconds,
    }, ensure_ascii=False, indent=2), encoding='utf-8'
)

metrics_summary = {
    'experiment': 'group_validation_flip',
    'best_epoch': best_epoch,
    'best_validation_metrics': best_validation_metrics,
    'epochs_ran': epochs_ran,
    'train_count': len(train_indices),
    'validation_count': len(val_indices),
    'test_count': len(test_data),
    'device': str(device),
}
(OUTPUT_DIR / 'metrics.json').write_text(
    json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Saved:', OUTPUT_DIR / 'model_best.pt')
print('Saved:', OUTPUT_DIR / 'model_last.pt')
print('Saved:', OUTPUT_DIR / 'config.json')
print('Saved:', OUTPUT_DIR / 'split.json')
print('Saved:', OUTPUT_DIR / 'history.json')
print('Saved:', OUTPUT_DIR / 'metrics.json')

## 16 绘制训练与验证损失曲线

用途：同时观察训练损失和验证损失随轮次的变化，辅助判断停滞或过拟合。训练损失来自增强图片，验证损失来自固定图片，不宜仅凭两条曲线的高低下结论。

依赖：训练后的 history。输出：learning_curves.png。

In [ ]:
# 16 绘制训练与验证损失曲线
# 用途和运行条件见上方说明单元。

plt.figure(figsize=(7, 4.5))
plt.plot(range(1, len(history['train_loss']) + 1), history['train_loss'], marker='o', label='Train loss')
plt.plot(range(1, len(history['val_loss']) + 1), history['val_loss'], marker='o', label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curves.png', dpi=160)
plt.show()

## 17 加载已保存的最佳模型

用途：从 OUTPUT_DIR/model_best.pt 恢复模型，用于后续诊断和预测；加载不会训练。

依赖：第 1、2、3、9 节及已有模型文件，不需要运行模型初始化或训练单元。这里单独创建 inference_model，不改变训练模型 model 和优化器的配对关系。

In [ ]:
# 17 加载已保存的最佳模型
# 用途和运行条件见上方说明单元。

CHECKPOINT_PATH = OUTPUT_DIR / 'model_best.pt'
assert CHECKPOINT_PATH.is_file(), f'找不到模型：{CHECKPOINT_PATH}'
best_checkpoint = torch.load(
    CHECKPOINT_PATH, map_location=device, weights_only=True
)
assert best_checkpoint['class_names'] == list(CLASS_NAMES), '模型类别顺序不匹配'
inference_model = ConcreteCrackCNN().to(device)
inference_model.load_state_dict(best_checkpoint['model_state_dict'])
inference_model.eval()
print('Loaded:', CHECKPOINT_PATH)
print('Best epoch:', best_checkpoint.get('epoch', '未记录'))

## 18 诊断最佳模型的训练集与验证集表现

用途：在固定预处理下比较训练集与验证集，查看分类指标和概率分布。

依赖：第 8、10、12、13、17 节。只做诊断，不更新参数。输出 diagnostics.json；不使用测试集选择方案。

In [ ]:
# 18 诊断最佳模型的训练集与验证集表现
# 用途和运行条件见上方说明单元。

diagnostics = {
    'best_epoch': best_checkpoint.get('epoch'),
    'train': summarize_current_split(inference_model, 'train', train_eval_loader),
    'validation': summarize_current_split(inference_model, 'validation', val_loader),
}
(OUTPUT_DIR / 'diagnostics.json').write_text(
    json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Saved:', OUTPUT_DIR / 'diagnostics.json')

## 19 查看验证集误报与漏检图片

用途：各展示最多 12 张误报和漏检图片，帮助发现阴影、纹理或细小裂缝等误判模式。

依赖：第 8、12、17 节。输出 validation_misclassified.png 和 misclassified_examples.json。示例按当前顺序取前几张，不代表全部错误的分布。

In [ ]:
# 19 查看验证集误报与漏检图片
# 用途和运行条件见上方说明单元。

validation_targets, validation_predictions, validation_probabilities = (
    collect_probability_diagnostics(inference_model, val_loader, device)
)
false_positive_indices = [
    position
    for position, (target, prediction) in enumerate(
        zip(validation_targets, validation_predictions)
    )
    if target == 0 and prediction == 1
]
false_negative_indices = [
    position
    for position, (target, prediction) in enumerate(
        zip(validation_targets, validation_predictions)
    )
    if target == 1 and prediction == 0
]

error_positions = false_positive_indices[:12] + false_negative_indices[:12]
figure, axes = plt.subplots(3, 8, figsize=(16, 7))
for axis in axes.flat:
    axis.axis('off')
for axis, position in zip(axes.flat, error_positions):
    dataset_index = val_indices[position]
    image_path, target = train_full.samples[dataset_index]
    with Image.open(image_path) as image:
        axis.imshow(image.convert('RGB'))
    predicted_name = CLASS_NAMES[validation_predictions[position]]
    target_name = CLASS_NAMES[target]
    probability = validation_probabilities[position]
    axis.set_title(
        f'true={target_name}\npred={predicted_name}\nP(cracked)={probability:.2f}',
        fontsize=8,
    )

figure.tight_layout()
figure.savefig(OUTPUT_DIR / 'validation_misclassified.png', dpi=160)
plt.show()

misclassified_summary = {
    'false_positive_count': len(false_positive_indices),
    'false_negative_count': len(false_negative_indices),
    'false_positive_examples': [
        train_full.samples[val_indices[position]][0]
        for position in false_positive_indices[:12]
    ],
    'false_negative_examples': [
        train_full.samples[val_indices[position]][0]
        for position in false_negative_indices[:12]
    ],
}
(OUTPUT_DIR / 'misclassified_examples.json').write_text(
    json.dumps(misclassified_summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('False positives:', len(false_positive_indices))
print('False negatives:', len(false_negative_indices))
print('Saved:', OUTPUT_DIR / 'validation_misclassified.png')
print('Saved:', OUTPUT_DIR / 'misclassified_examples.json')

## 20 最终测试集评估（可选）

用途：方案确定后，评估最佳模型在测试集上的表现。默认 RUN_TEST_EVALUATION=False，整本运行时会跳过。

启用后依赖第 8、11、17 节。输出 test_metrics.json；不要反复根据测试成绩选择权重、模型或阈值。

In [ ]:
# 20 最终测试集评估（可选）
# 用途和运行条件见上方说明单元。

if RUN_TEST_EVALUATION:
    targets, predictions = collect_predictions(inference_model, test_loader, device)
    test_metrics = calculate_metrics(targets, predictions)
    for name, value in test_metrics.items():
        print(f'{name}: {value:.4f}' if isinstance(value, float) else f'{name}: {value}')
    (OUTPUT_DIR / 'test_metrics.json').write_text(
        json.dumps(test_metrics, ensure_ascii=False, indent=2), encoding='utf-8'
    )
else:
    print("已跳过最终测试；方案确定后将 RUN_TEST_EVALUATION 改为 True。")

## 21 使用最佳模型识别自己的图片

用途：修改 sample_path 为实际图片路径，显示预测类别、两类概率和图片。

只预测的运行路线：第 1、2、3、4、9、17、21 节。概率是模型输出，不等于正确率；模型只做整张图片分类，不标注裂缝位置。

In [ ]:
# 21 使用最佳模型识别自己的图片
# 用途和运行条件见上方说明单元。

sample_path = r"C:\Users\72758\Desktop\科研\download.jpg"
with Image.open(sample_path) as image:
    rgb_image = image.convert('RGB')
    input_tensor = transform(rgb_image).unsqueeze(0).to(device)

inference_model.eval()
with torch.inference_mode():
    probabilities = torch.softmax(inference_model(input_tensor), dim=1)[0].cpu().numpy()

predicted_index = int(probabilities.argmax())
print('Image:', sample_path)
print('Prediction:', CLASS_NAMES[predicted_index])
print(f'P(uncracked): {probabilities[0]:.6f}')
print(f'P(cracked): {probabilities[1]:.6f}')

plt.figure(figsize=(5, 5))
plt.imshow(rgb_image)
plt.title(f'Prediction: {CLASS_NAMES[predicted_index]}')
plt.axis('off')
plt.show()

## 22 固定 32 张图片的学习能力检查（可选）

用途：从训练子集中各取 16 张图片，用新建的 small_model 重复训练，检查基本学习链路。重复 3 次，每次 50 轮；默认跳过，不属于正式实验。

启用 RUN_SMALL_SAMPLE_CHECK 后运行，依赖数据划分、模型定义、损失权重及训练/评估函数。结果保存为 small_sample_diagnostics.json，避免覆盖最佳模型的诊断记录。小样本达到 100% 不代表实际泛化性能。

In [ ]:
# 22 固定 32 张图片的学习能力检查（可选）
# 用途和运行条件见上方说明单元。

if RUN_SMALL_SAMPLE_CHECK:
    label_indices = {
        label: [index for index in train_indices if train_full.targets[index] == label]
        for label in range(2)
    }
    rng = random.Random(SEED)
    fixed_indices = []
    for label in range(2):
        selected = label_indices[label].copy()
        rng.shuffle(selected)
        fixed_indices.extend(selected[:16])
    rng.shuffle(fixed_indices)
    
    fixed_data = Subset(train_full, fixed_indices)
    fixed_loader = DataLoader(
        fixed_data,
        batch_size=32,
        shuffle=True,
        num_workers=NUM_WORKERS,
    )
    print('Fixed diagnostic labels:', Counter(train_full.targets[index] for index in fixed_indices))
    print('Fixed diagnostic paths:')
    for index in fixed_indices:
        print(CLASS_NAMES[train_full.targets[index]], train_full.samples[index][0])
    
    figure, axes = plt.subplots(4, 8, figsize=(16, 8))
    for axis, index in zip(axes.flat, fixed_indices):
        image_path, label = train_full.samples[index]
        with Image.open(image_path) as image:
            axis.imshow(image.convert('RGB'))
        axis.set_title(CLASS_NAMES[label])
        axis.axis('off')
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / 'fixed_32_samples.png', dpi=160)
    plt.show()
    
    small_sample_results = []
    for repetition in range(3):
        seed_everything(SEED + repetition)
        small_model = ConcreteCrackCNN().to(device)
        small_criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
        small_optimizer = optim.Adam(small_model.parameters(), lr=LEARNING_RATE)
        for _ in range(50):
            train_one_epoch(
                small_model,
                fixed_loader,
                small_criterion,
                small_optimizer,
                device,
            )
        small_loss, small_metrics = evaluate(
            small_model,
            fixed_loader,
            small_criterion,
            device,
        )
        result = {
            'repetition': repetition + 1,
            'loss': small_loss,
            **small_metrics,
        }
        small_sample_results.append(result)
        print(f'Fixed 32 samples repetition {repetition + 1}:', result)
    
    (OUTPUT_DIR / 'small_sample_diagnostics.json').write_text(
        json.dumps({'small_sample_results': small_sample_results}, indent=2),
        encoding='utf-8',
    )
    print('Saved:', OUTPUT_DIR / 'fixed_32_samples.png')
    print('Saved:', OUTPUT_DIR / 'small_sample_diagnostics.json')
else:
    print("已跳过固定 32 张图片检查；需要排错时再启用 RUN_SMALL_SAMPLE_CHECK。")

## 指标与文件速查

- Recall：实际裂缝中找出了多少；Precision：判为裂缝的图片中多少是真的。
- F1：综合 Precision 和 Recall；BA：裂缝召回率和无裂缝识别率的平均值。
- Train Loss：训练过程的加权损失；Validation Loss：验证集的加权损失，都不是错误百分比。
- model_best.pt：验证 F1 最高的模型；model_last.pt：训练最后一轮的模型；都需要结合网络定义加载。
- 当前训练/验证来自桥面目录，测试来自路面目录，应结合表面类型差异解释结果。

旧版重复诊断、早停后追加训练以及粘贴到开头的代码文本已从主流程移除，完整原件保存在 _backups；已有实验结果文件未更改。